# Hybrid Recommender + Evaluation — Multi-Item Orders Only

This notebook filters the recommender experiment to orders with more than one distinct product, while keeping the original project results unchanged.

## Setup & data loading

In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
from sqlalchemy import text, create_engine
from sqlalchemy.engine import URL

# Detect project root robustly, then work from it.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd
for candidate in [cwd] + list(cwd.parents):
    if (candidate / "M2_DWH").exists() or (candidate / "olist_dwh.dump").exists():
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

OUT_DIR = PROJECT_ROOT / "outputs" / "m5_multi_item"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Database connection: prefer DATABASE_URL, otherwise use PGPASSWORD, otherwise prompt.
database_url = os.environ.get("DATABASE_URL")
if database_url:
    engine = create_engine(database_url, future=True)
else:
    password = os.environ.get("PGPASSWORD") or getpass("Enter PostgreSQL password for user postgres: ")
    url = URL.create(
        drivername="postgresql+psycopg2",
        username="postgres",
        password=password,
        host="localhost",
        port=5432,
        database="olist_dwh"
    )
    engine = create_engine(url, future=True)

# Include order_id so we can filter multi-item orders inside this experiment.
interactions = pd.read_sql(text(
    'SELECT order_id, customer_id, product_id, orderdate FROM fact_orderitems'
), engine)
customers = pd.read_sql(text(
    'SELECT customer_id, customer_unique_id FROM dim_customers'
), engine)
products = pd.read_sql(text(
    'SELECT product_id, product_category_name_english, category_group_id FROM dim_products'
), engine)
prices = pd.read_sql(text(
    'SELECT product_id, AVG(price) AS avg_price FROM fact_orderitems GROUP BY product_id'
), engine)

# Load multi-item rules. Prefer with_segments if it exists, otherwise use ranked_rules_for_m5.
rules_candidates = [
    PROJECT_ROOT / 'outputs' / 'rules_multi_item' / 'ranked_rules_for_m5_with_segments.csv',
    PROJECT_ROOT / 'outputs' / 'rules_multi_item' / 'ranked_rules_for_m5.csv',
]
RULES_PATH = next((p for p in rules_candidates if p.exists()), None)
if RULES_PATH is None:
    raise FileNotFoundError('No multi-item rules found. Run M3_Association_Rules_Multi_Item.ipynb first.')

CLUSTERS_PATH = PROJECT_ROOT / 'outputs' / 'clustering' / 'customer_cluster_assignments.csv'
rules = pd.read_csv(RULES_PATH)
clusters = pd.read_csv(CLUSTERS_PATH) if CLUSTERS_PATH.exists() else pd.DataFrame(columns=['customer_id', 'cluster_id'])

# Make sure optional columns exist so the rest of the notebook is stable.
for col, default in [('condition', 'all'), ('segment_id', np.nan), ('algorithm', 'unknown')]:
    if col not in rules.columns:
        rules[col] = default

# -------------------------------------------------------------------
# MULTI-ITEM EXPERIMENT FILTER
# Keep only orders with more than one DISTINCT product.
# -------------------------------------------------------------------
original_rows = len(interactions)
order_product_counts = interactions.groupby('order_id')['product_id'].nunique()
multi_item_order_ids = set(order_product_counts[order_product_counts > 1].index)
all_orders = int(order_product_counts.shape[0])

interactions = interactions[interactions['order_id'].isin(multi_item_order_ids)].copy()
interactions = interactions.merge(customers, on='customer_id', how='left')
interactions['orderdate'] = pd.to_datetime(interactions['orderdate'])

multi_summary = pd.DataFrame({
    'metric': [
        'all_orders_in_fact_orderitems',
        'multi_item_orders_distinct_product_gt_1',
        'removed_orders',
        'multi_item_order_percentage',
        'original_fact_rows',
        'filtered_fact_rows',
        'filtered_unique_customers',
        'filtered_unique_products'
    ],
    'value': [
        all_orders,
        len(multi_item_order_ids),
        all_orders - len(multi_item_order_ids),
        (len(multi_item_order_ids) / all_orders * 100) if all_orders else 0,
        original_rows,
        len(interactions),
        interactions['customer_unique_id'].nunique(),
        interactions['product_id'].nunique()
    ]
})
multi_summary.to_csv(OUT_DIR / 'multi_item_interaction_summary.csv', index=False)

print('Project root:', PROJECT_ROOT)
print('Rules path:', RULES_PATH)
print(f'interactions after multi-item filter: {len(interactions):,} rows  '
      f'unique customers: {interactions["customer_unique_id"].nunique():,}  '
      f'products: {interactions["product_id"].nunique():,}')
print(f'rules: {len(rules)} rows  clusters: {len(clusters):,} rows')
print('\n=== Multi-item filter summary ===')
print(multi_summary.to_string(index=False))


Project root: C:\Users\moh75\olist-recommendation-system
Rules path: C:\Users\moh75\olist-recommendation-system\outputs\rules_multi_item\ranked_rules_for_m5_with_segments.csv
interactions after multi-item filter: 7,676 rows  unique customers: 3,167  products: 4,830
rules: 142 rows  clusters: 96,478 rows

=== Multi-item filter summary ===
                                 metric         value
          all_orders_in_fact_orderitems  96478.000000
multi_item_orders_distinct_product_gt_1   3197.000000
                         removed_orders  93281.000000
            multi_item_order_percentage      3.313709
                     original_fact_rows 110197.000000
                     filtered_fact_rows   7676.000000
              filtered_unique_customers   3167.000000
               filtered_unique_products   4830.000000


In [2]:
interactions.shape

(7676, 5)

In [3]:
# Customers who have bought more than 1 distinct product
multi_product_users = (
    interactions.groupby('customer_unique_id')['product_id']
                .nunique()
                .loc[lambda s: s > 1]
                .index
)

interactions[interactions['customer_unique_id'].isin(multi_product_users)] \
    .sort_values(['customer_unique_id', 'orderdate'])


,order_id,customer_id,product_id,orderdate,customer_unique_id
2021,44e608f2db00c74a1fe329de44416a4e,a81ebb9b32f102298c0c89635b4b3154,62984ea1bba7fcea1f5b57084d3bf885,2018-02-28,00053a61a98854899e70ed204dd4bafe
2022,44e608f2db00c74a1fe329de44416a4e,a81ebb9b32f102298c0c89635b4b3154,58727e154e8e85d84052cd22b0136c84,2018-02-28,00053a61a98854899e70ed204dd4bafe
5976,c6d61340bd8baeedca7cc8e7f7ec07e9,455f2e2988eaf87d7e2ba33b0a57969f,af0a917aec9cea3b353ece61a8825326,2017-08-17,000de6019bb59f34c099a907c151d855
5977,c6d61340bd8baeedca7cc8e7f7ec07e9,455f2e2988eaf87d7e2ba33b0a57969f,9e572ff4654f7064419d97a891a8b0fc,2017-08-17,000de6019bb59f34c099a907c151d855
4023,87440e08790d85796f5b8bc9f5ed2707,4b95f958af9c866353ae1108d8ebd023,38021cef829efa264df6f9d21c4df6e9,2018-07-26,000fbf0473c10fc1ab6f8d2d286ce20c
...,...,...,...,...,...
7050,ea134380898c6f487ff7113ecdd6319a,04e78263360f4c3aee74cc715c333608,b74e40daa954e92be8e5d3b342fa8863,2018-05-09,ffd6f65402f2bc47238ecd2bdc93e0d4
1230,2c43da70769acf988d27b050b189c7fe,7aa03a53e8c6489f3854470b6ebe9537,592962829d5a715304344e656e39108a,2017-12-27,fff7219c86179ca6441b8f37823ba3d3
1231,2c43da70769acf988d27b050b189c7fe,7aa03a53e8c6489f3854470b6ebe9537,056d012d264624accb7f73d31caee034,2017-12-27,fff7219c86179ca6441b8f37823ba3d3
3395,725cf8e9c24e679a8a5a32cb92c9ce1e,74be082247cd677a147d83ee670e9d53,c100e5fef1abb5e1c5054d1dac2d83ac,2017-06-08,fffcf5a5ff07b0908bd4e2dbc735a684


##   2   Chronological train/test split
Cutoff: 2018-04-01 (M2 handoff suggestion). Test users = customer_unique_ids present in both halves.

In [4]:
CUTOFF = pd.Timestamp('2018-04-01')

train = interactions[interactions['orderdate'] < CUTOFF].copy()
test  = interactions[interactions['orderdate'] >= CUTOFF].copy()

train_users    = set(train['customer_unique_id'].dropna().unique())
test_users_all = set(test['customer_unique_id'].dropna().unique())
test_users     = sorted(train_users & test_users_all)

train_user_items = train.groupby('customer_unique_id')['product_id'].apply(set).to_dict()
test_user_items  = test.groupby('customer_unique_id')['product_id'].apply(set).to_dict()

print(f'train: {len(train):,} rows, {len(train_users):,} unique customers')
print(f'test : {len(test):,} rows, {len(test_users_all):,} unique customers')
print(f'test users (in both train and test): {len(test_users):,}')
if len(test_users) == 0:
    raise RuntimeError('No test users with both train and test interactions   check cutoff.')

train: 4,994 rows, 2,084 unique customers
test : 2,682 rows, 1,091 unique customers
test users (in both train and test): 8


##   3   Evaluation harness (P@K, R@K, Hit Rate, Coverage)

In [5]:
def precision_at_k(predicted, actual, k=10):
    if not predicted: return 0.0
    return len(set(predicted[:k]) & actual) / k

def recall_at_k(predicted, actual, k=10):
    if not actual: return 0.0
    return len(set(predicted[:k]) & actual) / len(actual)

def hit_at_k(predicted, actual, k=10):
    return 1 if set(predicted[:k]) & actual else 0

def coverage(all_predictions, catalog_size):
    union = set()
    for preds in all_predictions:
        union.update(preds)
    return len(union) / catalog_size if catalog_size else 0.0

def run_eval(recommender_fn, name, test_users, test_user_items, catalog_size, k=10):
    t0 = time.time()
    rows, all_preds = [], []
    for u in test_users:
        preds = recommender_fn(u, k)
        all_preds.append(preds)
        actual = test_user_items.get(u, set())
        rows.append({
            'customer_unique_id': u,
            'precision_at_k': precision_at_k(preds, actual, k),
            'recall_at_k':    recall_at_k(preds, actual, k),
            'hit':            hit_at_k(preds, actual, k),
        })
    elapsed = time.time() - t0
    per_user = pd.DataFrame(rows)
    aggregate = {
        'system': name,
        'precision_at_10': per_user['precision_at_k'].mean(),
        'recall_at_10':    per_user['recall_at_k'].mean(),
        'hit_rate_at_10':  per_user['hit'].mean(),
        'coverage':        coverage(all_preds, catalog_size),
        'runtime_seconds': round(elapsed, 2),
        'n_users':         len(per_user),
    }
    print(f'{name:22s} P@10={aggregate["precision_at_10"]:.4f}  '
          f'R@10={aggregate["recall_at_10"]:.4f}  '
          f'HR={aggregate["hit_rate_at_10"]:.4f}  '
          f'Cov={aggregate["coverage"]:.4f}  '
          f't={elapsed:.1f}s  n={len(per_user)}')
    return aggregate, per_user

##   4   Non-ML baselines (most-popular, category-popular)

In [6]:
popular_items = train['product_id'].value_counts().index.tolist()

def most_popular_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    return [p for p in popular_items if p not in bought][:k]

prod_to_cat = products.set_index('product_id')['category_group_id'].to_dict()
train_with_cat = train.copy()
train_with_cat['category_group_id'] = train_with_cat['product_id'].map(prod_to_cat)

cat_to_popular = (
    train_with_cat.dropna(subset=['category_group_id'])
                  .groupby('category_group_id')['product_id']
                  .apply(lambda s: s.value_counts().index.tolist())
                  .to_dict()
)

user_top_cat = (
    train_with_cat.dropna(subset=['category_group_id'])
                  .groupby(['customer_unique_id', 'category_group_id'])
                  .size().reset_index(name='n')
                  .sort_values(['customer_unique_id', 'n'], ascending=[True, False])
                  .drop_duplicates('customer_unique_id')
                  .set_index('customer_unique_id')['category_group_id']
                  .to_dict()
)

def category_popular_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    cat = user_top_cat.get(user)
    if cat is None or cat not in cat_to_popular:
        return most_popular_recommend(user, k)
    return [p for p in cat_to_popular[cat] if p not in bought][:k]

prod_to_name = products.set_index('product_id')['product_category_name_english'].to_dict()

catalog_size = train['product_id'].nunique()
print(f'catalog size (train): {catalog_size:,}')
print('global top-5 popular products:')
for p in popular_items[:5]:
    print(f'  {p}  →  {prod_to_name.get(p, "?")}')


catalog size (train): 3,255
global top-5 popular products:
  99a4788cb24856965c36a24e339b6058  →  bed_bath_table
  36f60d45225e60c7da4558b070ce4b60  →  computers_accessories
  368c6c730842d78016ad823897a372db  →  garden_tools
  35afc973633aaeb6b877ff57b2793310  →  home_confort
  e53e557d5a159f5aa2c5e995dfdf244b  →  computers_accessories


##   5   Item-based collaborative filtering
Sparse user×item matrix → top-N item-item cosine similarity (truncated to 200 neighbours per item to fit in memory).

In [7]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

train_unique = train[['customer_unique_id', 'product_id']].dropna().drop_duplicates()
u2i = {u: i for i, u in enumerate(train_unique['customer_unique_id'].unique())}
p2i = {p: i for i, p in enumerate(train_unique['product_id'].unique())}
i2p = {i: p for p, i in p2i.items()}

rows = train_unique['customer_unique_id'].map(u2i).to_numpy()
cols = train_unique['product_id'].map(p2i).to_numpy()
data = np.ones(len(rows), dtype=np.float32)
ui_matrix = csr_matrix((data, (rows, cols)), shape=(len(u2i), len(p2i)))
iu_matrix = ui_matrix.T.tocsr()
print(f'user-item: {ui_matrix.shape}  nnz={ui_matrix.nnz:,}')

N_NEIGHBORS = 200

def topk_neighbors_sparse(item_matrix, n_neighbors=200, chunk=500):
    n = item_matrix.shape[0]
    neighbors = {}
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        block = cosine_similarity(item_matrix[start:end], item_matrix, dense_output=True)
        for r in range(end - start):
            block[r, start + r] = 0.0
        for r in range(end - start):
            row = block[r]
            if row.max() == 0:
                neighbors[start + r] = []
                continue
            k_eff = min(n_neighbors, n - 1)
            top = np.argpartition(-row, k_eff)[:k_eff]
            top = top[np.argsort(-row[top])]
            top = top[row[top] > 0]
            neighbors[start + r] = [(int(j), float(row[j])) for j in top]
    return neighbors

print('Computing item-item similarity (chunked)...')
_t0 = time.time()
item_neighbors = topk_neighbors_sparse(iu_matrix, n_neighbors=N_NEIGHBORS, chunk=500)
print(f'  done in {time.time()-_t0:.1f}s; edges={sum(len(v) for v in item_neighbors.values()):,}')

def cf_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    bought_idx = [p2i[p] for p in bought if p in p2i]
    if not bought_idx:
        return most_popular_recommend(user, k)
    bought_set = set(bought_idx)
    scores = {}
    for src in bought_idx:
        for (nbr, sim) in item_neighbors.get(src, []):
            if nbr in bought_set:
                continue
            scores[nbr] = scores.get(nbr, 0.0) + sim
    if not scores:
        return most_popular_recommend(user, k)
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return [i2p[i] for i, _ in top]

user-item: (2084, 3255)  nnz=4,523
Computing item-item similarity (chunked)...
  done in 0.1s; edges=5,314


##   6   Content-based fallback
Per-product features: one-hot(category_group_id) + one-hot(price_bucket from qcut on avg price). Item-item cosine similarity, top-200 neighbours per item.

In [8]:
prod_prices = prices.copy()
prod_prices['price_bucket'] = pd.qcut(
    prod_prices['avg_price'].rank(method='first'), q=5, labels=False, duplicates='drop'
)

catalog_items = train['product_id'].unique()
content_feat = (products[['product_id', 'category_group_id']]
                .merge(prod_prices[['product_id', 'price_bucket']], on='product_id', how='left'))
content_feat = content_feat[content_feat['product_id'].isin(catalog_items)].reset_index(drop=True)
content_feat['category_group_id'] = content_feat['category_group_id'].fillna(-1).astype(int)
content_feat['price_bucket']      = content_feat['price_bucket'].fillna(-1).astype(int)

cat_d   = pd.get_dummies(content_feat['category_group_id'], prefix='cat')
price_d = pd.get_dummies(content_feat['price_bucket'],      prefix='price')
feat_matrix = pd.concat([cat_d, price_d], axis=1).to_numpy(dtype=np.float32)
feat_sparse = csr_matrix(feat_matrix)

content_p2i = {p: i for i, p in enumerate(content_feat['product_id'].values)}
content_i2p = {i: p for p, i in content_p2i.items()}
print(f'content feature matrix: {feat_matrix.shape}')

print('Computing content item-item similarity (chunked)...')
_t0 = time.time()
content_neighbors = topk_neighbors_sparse(feat_sparse, n_neighbors=200, chunk=500)
print(f'  done in {time.time()-_t0:.1f}s')

def content_recommend(user, k=10):
    bought = train_user_items.get(user, set())
    bought_idx = [content_p2i[p] for p in bought if p in content_p2i]
    if not bought_idx:
        return most_popular_recommend(user, k)
    bought_set = set(bought_idx)
    scores = {}
    for src in bought_idx:
        for (nbr, sim) in content_neighbors.get(src, []):
            if nbr in bought_set:
                continue
            scores[nbr] = scores.get(nbr, 0.0) + sim
    if not scores:
        return most_popular_recommend(user, k)
    top = sorted(scores.items(), key=lambda x: -x[1])[:k]
    return [content_i2p[i] for i, _ in top]

content feature matrix: (3255, 16)
Computing content item-item similarity (chunked)...


  done in 0.2s


##   7   Rules-based systems (Apriori-only, FP-Growth-only)
Each uses the M3 rules CSV filtered to its own algorithm, on the user's **most-recent train purchase** as the query item. Topped up with most-popular when the rule index has no match (otherwise coverage is ~0%).

In [9]:
apriori_idx = (
    rules[(rules['algorithm'] == 'Apriori') & (rules['condition'] == 'all')]
        .sort_values('rank')
        .groupby('query_item')['recommended_item'].apply(list).to_dict()
)
fpgrowth_idx = (
    rules[(rules['algorithm'] == 'FP-Growth') & (rules['condition'] == 'all')]
        .sort_values('rank')
        .groupby('query_item')['recommended_item'].apply(list).to_dict()
)

user_last_product = (
    train.sort_values('orderdate')
         .drop_duplicates('customer_unique_id', keep='last')
         .set_index('customer_unique_id')['product_id'].to_dict()
)

def rules_recommend(user, rule_index, k=10):
    bought = train_user_items.get(user, set())
    last_p = user_last_product.get(user)
    recs = []
    if last_p is not None and last_p in rule_index:
        recs = [p for p in rule_index[last_p] if p not in bought][:k]
    if len(recs) < k:
        fillers = [p for p in popular_items if p not in bought and p not in recs]
        recs += fillers[:k - len(recs)]
    return recs[:k]

def apriori_recommend(user, k=10):
    return rules_recommend(user, apriori_idx, k)

def fpgrowth_recommend(user, k=10):
    return rules_recommend(user, fpgrowth_idx, k)

print(f'apriori_idx query_items:  {len(apriori_idx)}')
print(f'fpgrowth_idx query_items: {len(fpgrowth_idx)}')

apriori_idx query_items:  8
fpgrowth_idx query_items: 8


##   8   Hybrid via Reciprocal Rank Fusion (k=60)
Components: (a) holiday/seasonal rules   *empty, skipped*; (b) per-segment rules; (c) item-CF; (d) content fallback.
Rules CSV is **deduplicated** first to avoid the Apriori/FP-Growth/ECLAT triplication.

In [10]:
RRF_K = 60  

rules_dedup = (
    rules.assign(_priority=rules['algorithm'].map({'FP-Growth': 0, 'Apriori': 1, 'ECLAT-style': 2}).fillna(3))
         .sort_values(['query_item', 'recommended_item', 'condition', 'segment_id', '_priority'])
         .drop_duplicates(subset=['query_item', 'recommended_item', 'condition', 'segment_id'], keep='first')
         .drop(columns=['_priority'])
)
print(f'rules before dedup: {len(rules)}  after: {len(rules_dedup)}  query_items: {rules_dedup["query_item"].nunique()}')

segment_rules_idx = {}
seg_rows = rules_dedup[rules_dedup['condition'] == 'segment'].copy()
if len(seg_rows):
    seg_rows['segment_id'] = seg_rows['segment_id'].astype(int)
    for (seg, q), grp in seg_rows.sort_values('rank').groupby(['segment_id', 'query_item']):
        segment_rules_idx[(int(seg), q)] = grp['recommended_item'].tolist()
print(f'segment rules entries: {len(segment_rules_idx)}')

last_order_cid = (
    interactions.sort_values('orderdate')
                .drop_duplicates('customer_unique_id', keep='last')
                .set_index('customer_unique_id')['customer_id'].to_dict()
)
cid_to_cluster = clusters.set_index('customer_id')['cluster_id'].to_dict()

def user_cluster(user):
    cid = last_order_cid.get(user)
    if cid is None:
        return None
    c = cid_to_cluster.get(cid)
    return None if c is None else int(c)

def segment_rules_component(user, k=10):
    seg = user_cluster(user)
    last_p = user_last_product.get(user)
    if seg is None or last_p is None:
        return []
    recs = segment_rules_idx.get((seg, last_p), [])
    bought = train_user_items.get(user, set())
    return [p for p in recs if p not in bought][:k]

def rrf_fuse(ranked_lists, k_param=RRF_K, top_k=10):
    scores = {}
    for lst in ranked_lists:
        for rank, item in enumerate(lst, start=1):
            scores[item] = scores.get(item, 0.0) + 1.0 / (k_param + rank)
    return [item for item, _ in sorted(scores.items(), key=lambda x: -x[1])[:top_k]]

def hybrid_recommend(user, k=10):
    seg_list     = segment_rules_component(user, k)        # (b)
    cf_list      = cf_recommend(user, k)                    # (c)
    content_list = content_recommend(user, k)               # (d)
    fused = rrf_fuse([seg_list, cf_list, content_list], k_param=RRF_K, top_k=k)
    if len(fused) < k:
        fillers = [p for p in most_popular_recommend(user, k) if p not in fused]
        fused += fillers[:k - len(fused)]
    return fused[:k]

rules before dedup: 142  after: 94  query_items: 10
segment rules entries: 0


##   9   Run all 6 systems and write the comparison table

In [11]:
K = 10
recommenders = [
    ('Most-Popular',     most_popular_recommend),
    ('Category-Popular', category_popular_recommend),
    ('Apriori-only',     apriori_recommend),
    ('FP-Growth-only',   fpgrowth_recommend),
    ('CF-only',          cf_recommend),
    ('Hybrid (RRF)',     hybrid_recommend),
]

agg_rows, per_user_all = [], []
for name, fn in recommenders:
    agg, per_user = run_eval(fn, name, test_users, test_user_items, catalog_size, k=K)
    agg_rows.append(agg)
    per_user['system'] = name
    per_user_all.append(per_user)

comparison_df = pd.DataFrame(agg_rows)
comparison_df.to_csv(OUT_DIR / 'comparison_table.csv', index=False)
print('\nComparison table:')
print(comparison_df.to_string(index=False))

per_user_df = pd.concat(per_user_all, ignore_index=True).rename(
    columns={'precision_at_k': 'precision_at_10', 'recall_at_k': 'recall_at_10'}
)
per_user_df.to_csv(OUT_DIR / 'per_user_metrics.csv', index=False)

Most-Popular           P@10=0.0125  R@10=0.0417  HR=0.1250  Cov=0.0034  t=0.0s  n=8
Category-Popular       P@10=0.0000  R@10=0.0000  HR=0.0000  Cov=0.0138  t=0.0s  n=8
Apriori-only           P@10=0.0125  R@10=0.0417  HR=0.1250  Cov=0.0034  t=0.0s  n=8
FP-Growth-only         P@10=0.0125  R@10=0.0417  HR=0.1250  Cov=0.0034  t=0.0s  n=8
CF-only                P@10=0.0250  R@10=0.1250  HR=0.2500  Cov=0.0111  t=0.0s  n=8
Hybrid (RRF)           P@10=0.0125  R@10=0.0625  HR=0.1250  Cov=0.0218  t=0.0s  n=8

Comparison table:
          system  precision_at_10  recall_at_10  hit_rate_at_10  coverage  runtime_seconds  n_users
    Most-Popular           0.0125      0.041667           0.125  0.003379              0.0        8
Category-Popular           0.0000      0.000000           0.000  0.013825              0.0        8
    Apriori-only           0.0125      0.041667           0.125  0.003379              0.0        8
  FP-Growth-only           0.0125      0.041667           0.125  0.003379    

##   10   Significance testing (Wilcoxon + bootstrap CI + Cliff's delta)

In [12]:
from scipy.stats import wilcoxon

def cliffs_delta(x, y):
    n_gt = int(np.sum(x > y))
    n_lt = int(np.sum(x < y))
    n = len(x)
    return (n_gt - n_lt) / n if n else 0.0

def bootstrap_ci(diff, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    n = len(diff)
    means = np.empty(n_boot)
    for i in range(n_boot):
        sample = rng.choice(diff, size=n, replace=True)
        means[i] = sample.mean()
    return float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))

pivot = per_user_df.pivot_table(
    index='customer_unique_id', columns='system', values='precision_at_10', fill_value=0.0
)

HYBRID = 'Hybrid (RRF)'
sig_rows = []
for other in [c for c in pivot.columns if c != HYBRID]:
    x = pivot[HYBRID].to_numpy()
    y = pivot[other].to_numpy()
    diff = x - y
    if np.all(diff == 0):
        w_p = 1.0
    else:
        try:
            _, w_p = wilcoxon(x, y, zero_method='wilcox', alternative='two-sided')
            w_p = float(w_p)
        except ValueError:
            w_p = float('nan')
    ci_low, ci_high = bootstrap_ci(diff, n_boot=1000)
    sig_rows.append({
        'comparison':   f'Hybrid (RRF) vs {other}',
        'wilcoxon_p':   w_p,
        'mean_diff':    float(diff.mean()),
        'ci_low':       ci_low,
        'ci_high':      ci_high,
        'cliffs_delta': cliffs_delta(x, y),
        'n_pairs':      int(len(diff)),
    })

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(OUT_DIR / 'significance_results.csv', index=False)
print('\nSignificance results:')
print(sig_df.to_string(index=False))

# Sample recommendations for the paper
sample_users = list(test_users)[:20]
rec_rows = []
for u in sample_users:
    for name, fn in recommenders:
        recs = fn(u, K)
        rec_rows.append({'customer_unique_id': u, 'system': name, 'top_10': '|'.join(recs)})
pd.DataFrame(rec_rows).to_csv(OUT_DIR / 'recommendations_sample.csv', index=False)
print('\nWrote multi-item outputs to:', OUT_DIR)


Significance results:
                      comparison  wilcoxon_p  mean_diff  ci_low  ci_high  cliffs_delta  n_pairs
    Hybrid (RRF) vs Apriori-only         1.0     0.0000 -0.0375   0.0375         0.000        8
         Hybrid (RRF) vs CF-only         1.0    -0.0125 -0.0375   0.0000        -0.125        8
Hybrid (RRF) vs Category-Popular         1.0     0.0125  0.0000   0.0375         0.125        8
  Hybrid (RRF) vs FP-Growth-only         1.0     0.0000 -0.0375   0.0375         0.000        8
    Hybrid (RRF) vs Most-Popular         1.0     0.0000 -0.0375   0.0375         0.000        8

Wrote multi-item outputs to: C:\Users\moh75\olist-recommendation-system\outputs\m5_multi_item
